# Stage 08: Feature Engineering (Crosswalk-Corrected 5-Election Panel)

**Pipeline Stage:** Feature Engineering (Rebuilt)  
**Primary Input:** `data/processed/07_ward_master_dataset/ward_master_dataset_2000_2021_crosswalk_corrected.csv`  
**Output File:** `data/processed/08_feature_engineered_dataset/ward_features_2000_2021_crosswalk_corrected.csv`  

### Strict Leakage-Safe Construction
All historical electoral features (`PreviousTurnout`, `RegisteredVotersChange`, `RegistrationGrowth`, `SafeProvincialAverageTurnout`, and `SafeBelowProvincialAverageTurnout`) are strictly lagged to use only information observable *prior* to the target election cycle. No target-year information leaks into feature values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'Feature Engineering' else Path.cwd()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

master_path = PROCESSED_DIR / '07_ward_master_dataset' / 'ward_master_dataset_2000_2021_crosswalk_corrected.csv'
feat_df = pd.read_csv(master_path)
feat_df = feat_df.sort_values(['Ward', 'ElectionYear']).reset_index(drop=True)
print('Loaded master table:', feat_df.shape)


### Step 1: Consecutive Election Verification & Lag Features
Lags are only populated when the ward record originates from an immediately preceding consecutive election cycle.

In [2]:
feat_df['PreviousElectionYear'] = feat_df.groupby('Ward')['ElectionYear'].shift(1)
expected_prev = {2000: None, 2006: 2000, 2011: 2006, 2016: 2011, 2021: 2016}
feat_df['ExpectedPreviousYear'] = feat_df['ElectionYear'].map(expected_prev)
feat_df['ConsecutiveElection'] = (feat_df['PreviousElectionYear'] == feat_df['ExpectedPreviousYear'])

# Safe electoral features
feat_df['PreviousTurnout'] = np.where(feat_df['ConsecutiveElection'], feat_df.groupby('Ward')['TurnoutRate'].shift(1), np.nan)
feat_df['PreviousRegistered'] = np.where(feat_df['ConsecutiveElection'], feat_df.groupby('Ward')['RegisteredVoters'].shift(1), np.nan)
feat_df['RegisteredVotersChange'] = feat_df['RegisteredVoters'] - feat_df['PreviousRegistered']
feat_df['RegistrationGrowth'] = (feat_df['RegisteredVotersChange'] / feat_df['PreviousRegistered']) * 100

# 2026 Naive Persistence Projection (most recent observed 2021 turnout)
turnout_2021_map = feat_df[feat_df['ElectionYear'] == 2021].set_index('Ward')['TurnoutRate'].to_dict()
feat_df['BaselinePredictedTurnout2026'] = feat_df['Ward'].map(turnout_2021_map)

# Safe Lagged Provincial Benchmarks
prov_means = feat_df.groupby('ElectionYear')['TurnoutRate'].mean().to_dict()
feat_df['ProvincialAverageTurnout'] = feat_df['ElectionYear'].map(prov_means)
feat_df['BelowProvincialAverageTurnout'] = feat_df['TurnoutRate'] < feat_df['ProvincialAverageTurnout']

feat_df['SafeProvincialAverageTurnout'] = feat_df['PreviousElectionYear'].map(prov_means)
feat_df['SafeBelowProvincialAverageTurnout'] = np.where(
    feat_df['PreviousTurnout'].notna(),
    (feat_df['PreviousTurnout'] < feat_df['SafeProvincialAverageTurnout']).astype(float),
    np.nan
)

out_feat_path = PROCESSED_DIR / '08_feature_engineered_dataset' / 'ward_features_2000_2021_crosswalk_corrected.csv'
feat_df.to_csv(out_feat_path, index=False)
print(f'Saved feature set: {out_feat_path.name} with shape {feat_df.shape}')
